# Prática 1 — Classificação: regressão logística e árvore de decisão

**Tempo:** 30 minutos · **Público:** iniciantes · **Ambiente:** Google Colab (CPU)

**Pergunta:** Como reconhecer padrões associados a um desfecho binário?

> Dados sintéticos e textos didáticos. Não representam evidência empírica nem recomendações clínicas.

### Como utilizar
Execute as células na ordem, uma de cada vez. Leia a explicação antes de clicar em ▶. Ao final, experimente alterar apenas os parâmetros indicados.

## Passo 1 — Importar bibliotecas

Usaremos apenas pacotes já disponíveis no Colab.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import confusion_matrix, recall_score, precision_score

## Passo 2 — Criar registros didáticos

Simulamos 400 registros com duas características e um desfecho binário fictício. Não são estimativas reais de mortalidade.

In [ ]:
rng = np.random.default_rng(42)
n = 400

idade_gestacional = rng.integers(28, 42, n)
consultas_prenatais = rng.integers(0, 11, n)

# Regra didática: valores menores tendem a gerar mais eventos.
pontuacao = 1.7 - 0.12 * idade_gestacional - 0.08 * consultas_prenatais
probabilidade = 1 / (1 + np.exp(-pontuacao))
evento = rng.binomial(1, probabilidade)

dados = pd.DataFrame({
    "idade_gestacional": idade_gestacional,
    "consultas_prenatais": consultas_prenatais,
    "evento": evento
})
print(dados.head())
print("Eventos observados:", dados["evento"].sum())

## Passo 3 — Separar dados para aprender e testar

Amostras de treino e teste são separadas para evitar medir apenas a memorização do modelo.

In [ ]:
X = dados[["idade_gestacional", "consultas_prenatais"]]
y = dados["evento"]

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
print("Treino:", len(X_treino), "Teste:", len(X_teste))

## Passo 4 — Comparar dois modelos

Ambos recebem as mesmas variáveis; o objetivo é comparar comportamento e erros, não declarar um vencedor universal.

In [ ]:
modelos = {
    "Regressão logística": LogisticRegression(max_iter=1000),
    "Árvore de decisão": DecisionTreeClassifier(
        max_depth=3, random_state=42
    )
}

for nome, modelo in modelos.items():
    modelo.fit(X_treino, y_treino)
    previsao = modelo.predict(X_teste)

    print("\n", nome)
    print("Matriz de confusão:\n", confusion_matrix(y_teste, previsao))
    print("Sensibilidade:", round(recall_score(y_teste, previsao), 2))
    print("Precisão:", round(precision_score(
        y_teste, previsao, zero_division=0
    ), 2))

## Passo 5 — Visualizar a árvore

Observe os limiares encontrados pelo algoritmo. Eles são artificiais e não são critérios clínicos.

In [ ]:
arvore = modelos["Árvore de decisão"]

plt.figure(figsize=(11, 5))
plot_tree(
    arvore,
    feature_names=X.columns,
    class_names=["Sem evento", "Com evento"],
    filled=True,
    rounded=True,
    fontsize=9
)
plt.title("Árvore de decisão — exemplo sintético")
plt.show()

## Agora é sua vez

Altere max_depth de 3 para 1 e depois para 5. A árvore muda? Sensibilidade e precisão melhoram sempre?

## Antes de usar dados reais

Documente a origem, a qualidade, o período, a unidade de observação e as condições de uso dos dados. Uma boa previsão ou resposta automática não demonstra causalidade nem dispensa validação.